In [46]:
# AI POWERED FUNDAMENTAL STOCK ANALYSIS USING RAG

# The objective of this project is to build an AI-powered financial analysis system that can analyze a selected stock
# using financial data from Yahoo Finance and provide fundamental analysis through a
# Retrieval-Augmented Generation (RAG) architecture.

# The project should demonstrate how structured financial data can be transformed
# into a searchable knowledge base using FAISS, and how Google Gemini LLM can use the retrieved financial
# context to generate meaningful, data-grounded fundamental analysis.

In [47]:
!pip install faiss-cpu
!pip install google-generativeai
!pip install pandas

In [48]:
!pip install yfinance

In [49]:
pip install langchain-text-splitters


In [50]:
import pandas as pd
import numpy as pd
from google import genai  ###communication with Gemini API
import faiss              #vector store
import yfinance as yf     #stock data
from sentence_transformers import SentenceTransformer #sentence embedding
from langchain_text_splitters import RecursiveCharacterTextSplitter ## text splitter

In [51]:
ticker = "META"
stock = yf.Ticker(ticker)

In [52]:
historical_data = stock.history(period = "6mo")
print(historical_data.tail())
print("-------------------------------------------------------------------")
print(historical_data.head())

                                 Open        High         Low       Close  \
Date                                                                        
2026-10-05 00:00:00-04:00  727.710022  746.700012  727.119995  741.900024   
2026-10-06 00:00:00-04:00  746.729980  747.599976  736.690002  738.880005   
2026-10-07 00:00:00-04:00  738.289978  738.289978  720.150024  721.309998   
2026-10-08 00:00:00-04:00  724.539978  724.950012  711.679993  720.890015   
2026-10-09 00:00:00-04:00  726.510010  728.070007  716.200012  718.669983   

                             Volume  Dividends  Stock Splits  
Date                                                          
2026-10-05 00:00:00-04:00  15215300        0.0           0.0  
2026-10-06 00:00:00-04:00  12563100        0.0           0.0  
2026-10-07 00:00:00-04:00  13231600        0.0           0.0  
2026-10-08 00:00:00-04:00  16041100        0.0           0.0  
2026-10-09 00:00:00-04:00  12605100        0.0           0.0  
-------------------

In [53]:
company_info = stock.info
print(company_info)

{'address1': '1 Meta Way', 'city': 'Menlo Park', 'state': 'CA', 'zip': '94025', 'country': 'United States', 'phone': '650 543 4800', 'website': 'https://www.meta.com', 'industry': 'Internet Content & Information', 'industryKey': 'internet-content-information', 'industryDisp': 'Internet Content & Information', 'sector': 'Communication Services', 'sectorKey': 'communication-services', 'sectorDisp': 'Communication Services', 'longBusinessSummary': "Meta Platforms, Inc. engages in the development of products that enable people to connect and share with friends and family through mobile devices, personal computers, virtual reality (VR) headsets, and AI glasses in the United States, Canada, Europe, Asia-Pacific, and internationally. It operates through two segments, Family of Apps (FoA) and Reality Labs (RL). The FoA segment offers Facebook, which enables people to build community through feed, reels, stories, groups, marketplace, and other; Instagram that brings people closer through Instag

In [54]:
news = stock.news
print(news)

[]


In [55]:
for item in news[:5]:
  print(item['content']['title'])
  print(item['content']['summary'])

In [56]:
### Create a Financial Knowledge Base
### Combine Historical Data + Company Info + News
documents = []
company_text = f"""
  Industry : {company_info['industry']}
  Sector : {company_info['sector']}
  Description : {company_info['longBusinessSummary']}
"""
documents.append(company_text)
print(documents)


["\n  Industry : Internet Content & Information\n  Sector : Communication Services\n  Description : Meta Platforms, Inc. engages in the development of products that enable people to connect and share with friends and family through mobile devices, personal computers, virtual reality (VR) headsets, and AI glasses in the United States, Canada, Europe, Asia-Pacific, and internationally. It operates through two segments, Family of Apps (FoA) and Reality Labs (RL). The FoA segment offers Facebook, which enables people to build community through feed, reels, stories, groups, marketplace, and other; Instagram that brings people closer through Instagram feed, stories, reels, live, and messaging; Messenger, a messaging application for people to connect with friends, family, communities, and businesses across platforms and devices through text, audio, and video calls; Meta AI, an assistant that's available across apps, as a stand-alone app, on AI glasses, and on the web; Threads, an application 

In [57]:
for item in news[:5]:
  news_text = f""""
  Title : {item['content']['title']}
  Summary : {item['content']['summary']}
  """
  documents.append(news_text)
print(documents)

["\n  Industry : Internet Content & Information\n  Sector : Communication Services\n  Description : Meta Platforms, Inc. engages in the development of products that enable people to connect and share with friends and family through mobile devices, personal computers, virtual reality (VR) headsets, and AI glasses in the United States, Canada, Europe, Asia-Pacific, and internationally. It operates through two segments, Family of Apps (FoA) and Reality Labs (RL). The FoA segment offers Facebook, which enables people to build community through feed, reels, stories, groups, marketplace, and other; Instagram that brings people closer through Instagram feed, stories, reels, live, and messaging; Messenger, a messaging application for people to connect with friends, family, communities, and businesses across platforms and devices through text, audio, and video calls; Meta AI, an assistant that's available across apps, as a stand-alone app, on AI glasses, and on the web; Threads, an application 

In [58]:
##### Historical Data
historical_data_text = historical_data.tail(30).to_string()
documents.append(historical_data_text)
print(documents)

["\n  Industry : Internet Content & Information\n  Sector : Communication Services\n  Description : Meta Platforms, Inc. engages in the development of products that enable people to connect and share with friends and family through mobile devices, personal computers, virtual reality (VR) headsets, and AI glasses in the United States, Canada, Europe, Asia-Pacific, and internationally. It operates through two segments, Family of Apps (FoA) and Reality Labs (RL). The FoA segment offers Facebook, which enables people to build community through feed, reels, stories, groups, marketplace, and other; Instagram that brings people closer through Instagram feed, stories, reels, live, and messaging; Messenger, a messaging application for people to connect with friends, family, communities, and businesses across platforms and devices through text, audio, and video calls; Meta AI, an assistant that's available across apps, as a stand-alone app, on AI glasses, and on the web; Threads, an application 

In [59]:
print(documents[0])


  Industry : Internet Content & Information
  Sector : Communication Services
  Description : Meta Platforms, Inc. engages in the development of products that enable people to connect and share with friends and family through mobile devices, personal computers, virtual reality (VR) headsets, and AI glasses in the United States, Canada, Europe, Asia-Pacific, and internationally. It operates through two segments, Family of Apps (FoA) and Reality Labs (RL). The FoA segment offers Facebook, which enables people to build community through feed, reels, stories, groups, marketplace, and other; Instagram that brings people closer through Instagram feed, stories, reels, live, and messaging; Messenger, a messaging application for people to connect with friends, family, communities, and businesses across platforms and devices through text, audio, and video calls; Meta AI, an assistant that's available across apps, as a stand-alone app, on AI glasses, and on the web; Threads, an application for t

In [60]:
#### Text Splitting - Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 0)
chunks = text_splitter.create_documents(documents)
print(chunks)

[Document(metadata={}, page_content='Industry : Internet Content & Information\n  Sector : Communication Services'), Document(metadata={}, page_content="Description : Meta Platforms, Inc. engages in the development of products that enable people to connect and share with friends and family through mobile devices, personal computers, virtual reality (VR) headsets, and AI glasses in the United States, Canada, Europe, Asia-Pacific, and internationally. It operates through two segments, Family of Apps (FoA) and Reality Labs (RL). The FoA segment offers Facebook, which enables people to build community through feed, reels, stories, groups, marketplace, and other; Instagram that brings people closer through Instagram feed, stories, reels, live, and messaging; Messenger, a messaging application for people to connect with friends, family, communities, and businesses across platforms and devices through text, audio, and video calls; Meta AI, an assistant that's available across apps, as a stand

In [61]:
print(len(chunks))

8


In [62]:
print(chunks[7])

page_content='2026-10-05 00:00:00-04:00  727.710022  746.700012  727.119995  741.900024  15215300      0.000           0.0
2026-10-06 00:00:00-04:00  746.729980  747.599976  736.690002  738.880005  12563100      0.000           0.0
2026-10-07 00:00:00-04:00  738.289978  738.289978  720.150024  721.309998  13231600      0.000           0.0
2026-10-08 00:00:00-04:00  724.539978  724.950012  711.679993  720.890015  16041100      0.000           0.0
2026-10-09 00:00:00-04:00  726.510010  728.070007  716.200012  718.669983  12605100      0.000           0.0'


In [63]:
#### Embeddings of the Chunks

In [64]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
text = [doc.page_content for doc in chunks]
embeddings = embedding_model.encode(text)
print(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[[ 0.04287845 -0.07941861 -0.08056921 ... -0.09427074  0.02531164
   0.02336515]
 [ 0.00550962 -0.06751323 -0.04785044 ...  0.07368374  0.02049647
   0.01000524]
 [-0.01111238 -0.04450534 -0.06306999 ...  0.0005538   0.0251898
   0.00559459]
 ...
 [ 0.07300588 -0.06003964  0.00689647 ... -0.03707359 -0.12017734
  -0.03585283]
 [ 0.06695403 -0.07501973 -0.01643787 ... -0.02837    -0.1066349
  -0.0136219 ]
 [ 0.05784839 -0.07190382 -0.01650027 ... -0.0228515  -0.10075374
  -0.0179248 ]]


In [65]:
print(embeddings.shape)

(8, 384)


In [66]:
import faiss
import numpy as np

# Convert the embeddings list into a numpy array of type float32 (required by FAISS)
embedding_matrix = np.array(embeddings).astype("float32")

# Get the dimension of the embeddings (which is 384)
dimension = embedding_matrix.shape[1]

# Initialize a FAISS index using L2 (Euclidean) distance
faiss_index = faiss.IndexFlatL2(dimension)

faiss_index.add(embedding_matrix)

print(f"Successfully built FAISS index with {faiss_index.ntotal} vectors.")

Successfully built FAISS index with 8 vectors.


In [67]:
def retrieve_financial_context(query, k=3):
    # 1. Embed the user's query
    query_vector = embedding_model.encode([query]).astype("float32")

    # 2. Search the FAISS index for the 'k' most similar vectors
    distances, indices = faiss_index.search(query_vector, k)

    # 3. Retrieve the actual text for the matching indices

    retrieved_texts = [chunks[idx].page_content for idx in indices[0] if idx < len(chunks)]

    # 4. Combine the retrieved chunks into a single string
    return "\n\n---\n\n".join(retrieved_texts)

# Testing the retrieval system
test_query = "What does Meta's Reality Labs segment do?"
retrieved_data = retrieve_financial_context(test_query, k=2)
print("Retrieved Context:\n", retrieved_data)

Retrieved Context:
 application that is used by people and businesses to communicate and transact. The RL segment provides virtual and augmented reality products, including consumer hardware, software, and content that help people feel connected, as well as Meta Quest devices that enable social experiences across gaming, fitness, entertainment, and more. The segment also includes wearables such as AI glasses like Ray Ban Meta and Oakley Meta glasses; and the Meta Ray Ban Display, which combines AI glasses with an integrated lens display and the Meta Neural Band, a wrist worn device using electromyography that lets people control their AI glasses through neuromuscular signals. Meta Platforms, Inc. has a collaboration with Microsoft Corporation, NVIDIA Corporation, Advanced Micro Devices, Inc., Broadcom Inc., and OpenAI, L.L.C. The company was formerly known as Facebook, Inc. and changed its name to Meta Platforms, Inc. in October 2021. The company was incorporated in 2004 and is headqua

In [68]:
import os
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

def generate_financial_analysis(query):
    # 1. Retrieve relevant financial data based on the query
    context = retrieve_financial_context(query, k=3)

    # 2. Build the augmented prompt
    prompt = f"""
    You are an expert AI Financial Analyst.
    Please answer the user's question using ONLY the provided financial context below.
    If the context does not contain the answer, explicitly state that the data is unavailable.

    Financial Context:
    {context}

    User Question:
    {query}

    Analysis:
    """

    # 3. Generate the response using the latest supported model
    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents=prompt
    )

    return response.text

# --- Final Test ---
user_question = "Provide a brief summary of Meta's business segments and any recent AI news."
analysis = generate_financial_analysis(user_question)

print("QUESTION:", user_question)
print("\nAI ANALYSIS:\n", analysis)

QUESTION: Provide a brief summary of Meta's business segments and any recent AI news.

AI ANALYSIS:
 Based on the provided context, here is a summary of Meta's business segments and the available information regarding AI:

### **Business Segments**
Meta operates through two primary segments:
1. **Family of Apps (FoA):** Encompasses Meta's social networking, messaging, and communication platforms, including Facebook, Instagram, Messenger, WhatsApp, Threads, and the Meta AI assistant.
2. **Reality Labs (RL):** Focuses on augmented and virtual reality products, including consumer hardware, software, and content. Key offerings include Meta Quest devices, wearables like Ray-Ban Meta and Oakley Meta AI glasses, the Meta Ray Ban Display, and the Meta Neural Band.

---

### **Recent AI News**
* **Data Unavailable:** The provided context does not contain any recent AI news. It only notes existing AI-related offerings (such as the Meta AI assistant, Ray-Ban Meta/Oakley Meta AI glasses, and Meta 

In [73]:
# Add financial ratios and statements
ratios_doc = f"""
Key Fundamental Ratios and Metrics for {ticker}:
- Market Cap: ${company_info.get('marketCap', 'N/A'):,}
- Trailing P/E: {company_info.get('trailingPE', 'N/A')}
- Forward P/E: {company_info.get('forwardPE', 'N/A')}
- PEG Ratio: {company_info.get('pegRatio', 'N/A')}
- Profit Margins: {company_info.get('profitMargins', 0) * 100:.2f}%
- Operating Margins: {company_info.get('operatingMargins', 0) * 100:.2f}%
- Return on Equity (ROE): {company_info.get('returnOnEquity', 0) * 100:.2f}%
- Total Revenue: ${company_info.get('totalRevenue', 'N/A'):,}
- Revenue Growth: {company_info.get('revenueGrowth', 0) * 100:.2f}%
- Diluted EPS: {company_info.get('trailingEps', 'N/A')}
- Total Debt: ${company_info.get('totalDebt', 'N/A'):,}
- Total Cash: ${company_info.get('totalCash', 'N/A'):,}
- Debt to Equity: {company_info.get('debtToEquity', 'N/A')}
- Free Cash Flow: ${company_info.get('freeCashflow', 'N/A'):,}
"""
documents.append(ratios_doc)

# Financial statements
if hasattr(stock, 'financials') and not stock.financials.empty:
    documents.append(f"Annual Income Statement for {ticker}:\n" + stock.financials.iloc[:, :2].to_string())

if hasattr(stock, 'balance_sheet') and not stock.balance_sheet.empty:
    documents.append(f"Annual Balance Sheet for {ticker}:\n" + stock.balance_sheet.iloc[:, :2].to_string())

if hasattr(stock, 'cashflow') and not stock.cashflow.empty:
    documents.append(f"Annual Cash Flow Statement for {ticker}:\n" + stock.cashflow.iloc[:, :2].to_string())

# Re-chunk and update vector store
chunks = text_splitter.create_documents(documents)
chunk_texts = [doc.page_content for doc in chunks]

embeddings = embedding_model.encode(chunk_texts)
embedding_matrix = np.array(embeddings).astype("float32")

# Update faiss_index so the retrieval function uses all chunks
faiss_index = faiss.IndexFlatL2(embedding_matrix.shape[1])
faiss_index.add(embedding_matrix)

print(f"Total documents: {len(documents)}")
print(f"Total chunks: {len(chunks)}")
print(f"Vectors in index: {faiss_index.ntotal}")

Total documents: 10
Total chunks: 44
Vectors in index: 44


In [70]:
def generate_financial_analysis(query, k=4):
    # 1. Retrieve top-k nearest matching chunks from FAISS
    context = retrieve_financial_context(query, k=k)

    # 2. Prompt engineering
    system_prompt = (
        "You are an expert AI Financial Analyst evaluating the stock. "
        "Answer the user's query strictly using the provided financial context. "
        "Include exact numbers, financial ratios, and growth percentages where available. "
        "If a specific metric is not present in the context, explicitly state that it is unavailable."
    )
    user_prompt = f"""Financial Context:
{context}

User Question: {query}

Structured Financial Analysis:"""

    # 3. Call Gemini model
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=f"{system_prompt}\n\n{user_prompt}"
    )
    return response.text

In [71]:
import time

for idx, q in enumerate(queries, 1):
    print(f"\n{'='*75}")
    print(f"CAPSTONE TEST CASE {idx}: {q}")
    print(f"{'='*75}\n")

    try:

        analysis = generate_financial_analysis(q, k=4)
        print(analysis)
    except Exception as e:

        print(f"\n[Server busy. Retrying in 5 seconds...]")
        time.sleep(5)
        analysis = generate_financial_analysis(q, k=4)
        print(analysis)


    time.sleep(3)


CAPSTONE TEST CASE 1: Analyze Meta's revenue, revenue growth rate, profit margins, and EPS.

### Structured Financial Analysis: Meta Platforms, Inc.

Based strictly on the provided financial context, here is the analysis of the requested metrics:

* **Revenue:** **Unavailable** (The provided context contains company description details, sector/industry classification, and historical stock trading data, but does not provide revenue figures).
* **Revenue Growth Rate:** **Unavailable** (Historical and current financial statement data required to calculate growth rates are not provided in the context).
* **Profit Margins (Gross, Operating, Net):** **Unavailable** (Income statement metrics such as gross profit, operating income, or net income are not present in the context).
* **Earnings Per Share (EPS):** **Unavailable** (EPS figures are not included in the provided text).

---

### Contextual Financial Data Available
While income statement and performance metrics are not present, the con

In [72]:
# Interactive Query Demo
print("=" * 60)
print(f"AI Financial Analyst for {ticker} is ready!")
print("=" * 60)

user_q = input("\nEnter your financial question (e.g., 'What are Meta's P/E ratios and cash flow?'): ")

if user_q.strip():
    print("\nGenerating Analysis...\n")
    print(generate_financial_analysis(user_q, k=4))

AI Financial Analyst for META is ready!

Enter your financial question (e.g., 'What are Meta's P/E ratios and cash flow?'): what are meta's P/E ratio

Generating Analysis...

Based on the provided financial context, Meta's **P/E (Price-to-Earnings) ratio is unavailable**. 

The provided data contains historical trading prices, volume, dividends, stock splits, and a company business description, but it does not include earnings per share (EPS) or valuation ratios.


# Project Summary & Documentation

### 1. Architecture Overview
1. **Data Ingestion:** I collected Meta's company profile, 6-month price history, key financial ratios, news headlines, and annual financial statements using the `yfinance` library.
2. **Text Chunking:** I cleaned and split the combined financial documents into chunks using LangChain's `RecursiveCharacterTextSplitter` (chunk size: 1000, overlap: 0).
3. **Embeddings:** Converted each text chunk into 384-dimensional vector embeddings using the `all-MiniLM-L6-v2` model from `sentence-transformers`.
4. **Vector Database:** Built a FAISS index (`IndexFlatL2`) and loaded all 44 vectors to allow fast similarity searching based on Euclidean distance.
5. **Retrieval & Augmentation:** When a question is asked, it gets converted to a vector, searches FAISS for the top 4 most relevant chunks, and adds that data into the prompt as context.
6. **Generation:** Sent the question and retrieved financial context to Google Gemini (`gemini-3.8-flash`) with clear instructions to only answer using the provided numbers to avoid guessing.

### 2. Technologies Used
- **Environment:** Python 3 (Google Colab)
- **Data Fetching:** `yfinance`, `requests`
- **Chunking:** `langchain-text-splitters`
- **Embeddings:** `sentence-transformers` (`all-MiniLM-L6-v2`)
- **Vector DB:** `faiss-cpu`
- **LLM API:** `google-genai` (`gemini-3.8-flash`)

### 3. Assumptions & Limitations
- **Static Snapshot:** The data represents a snapshot taken at runtime; live real-time stock ticks require continuous streaming.
- **Yahoo Finance Limits:** To avoid being blocked by Yahoo's rate limits, custom headers and delay/retry blocks were implemented.
- **Strict Data Grounding:** If a specific metric (like EPS or news) isn't inside the retrieved chunks, the model states that the information is unavailable rather than hallucinating figures.

### 4. Key Results
- The system correctly extracted and analyzed Meta's revenue ($228.2B), growth (28%), profit margins (29.83%), and balance sheet figures without making up numbers.
- The retrieval system pulled both news and financial statements, providing a balanced breakdown of Meta's growth versus its heavy CapEx and rising debt risks.